In [3]:
INPUT_CSV  = "/kaggle/input/datasets/sauraviitjapdcl/duomind-evaluation-data/evaluation_results.csv"
OUTPUT_CSV = f"/kaggle/working/judge_{MODEL_KEY}.csv"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"


In [19]:
df_full = pd.read_csv(INPUT_CSV)
#df = df_full[df_full["routed_engine"] == "Deep Thinker"].reset_index(drop=True)
print(f"Judging {len(df)} Deep Thinker rows only")

Judging 255 Deep Thinker rows only


In [6]:
# ── CHANGE THIS ONLY ─────────────────────────────────────────────
#MODEL_KEY  = "qwen"
#model_name = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_KEY  = "mistral"
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
INPUT_CSV  = "/kaggle/input/datasets/sauraviitjapdcl/duomind-evaluation-data/evaluation_results.csv"
OUTPUT_CSV = f"/kaggle/working/judge_{MODEL_KEY}.csv"

import os, re, gc, time, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

JUDGE_PROMPT = """You are evaluating a question-answering system on policy documents.

Question: {question}

Ground Truth Answer: {ground_truth}

System Answer: {system_answer}

Does the system answer convey the same information as the ground truth?
- Score 1.0 if the system answer is correct and contains the key information from the ground truth
- Score 0.5 if the system answer is partially correct or contains the right idea but is incomplete
- Score 0.0 if the system answer is wrong or does not address the question

Reply with ONLY one of: 1.0, 0.5, or 0.0
Score:"""

def parse_score(text):
    text = text.strip()
    for val in ["1.0", "0.5", "0.0", "1", "0"]:
        if text.startswith(val):
            return float(val)
    match = re.search(r"\b(1\.0|0\.5|0\.0|1|0)\b", text)
    return float(match.group(1)) if match else None

def clean_answer(text):
    text = str(text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:600]

def judge_row(row):
    prompt = JUDGE_PROMPT.format(
        question      = str(row.get("question", "")).strip(),
        ground_truth  = str(row.get("ground_truth_answer", "")).strip(),
        system_answer = clean_answer(row.get("system_answer", "")),
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False
    )
    inputs    = tokenizer(text, return_tensors="pt").to(DEVICE)
    input_ids = inputs["input_ids"]
    with torch.no_grad():
        output = model.generate(
            input_ids, max_new_tokens=16, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()
    return parse_score(raw), raw

# ── DELETE OLD FILE ───────────────────────────────────────────────
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)
    print("Deleted old file")

# ── CLEAR + LOAD ─────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    
    device_map="auto", trust_remote_code=True,
)
model.eval()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

# ── FILTER: Deep Thinker only ─────────────────────────────────────
df_full = pd.read_csv(INPUT_CSV)
df= df_full
#df = df_full[df_full["routed_engine"] == "Deep Thinker"].reset_index(drop=True)
#print(f"Judging {len(df)} Deep Thinker rows")

# ── QUICK TEST ────────────────────────────────────────────────────
row = df.iloc[0]
score, raw = judge_row(row)
print(f"Test Q : {row['question'][:70]}")
print(f"Test GT: {row['ground_truth_answer'][:70]}")
print(f"Score  : {score} | Raw: {raw}")

# ── MAIN LOOP ────────────────────────────────────────────────────
results, t0 = [], time.time()

for i, row in df.iterrows():
    try:
        score, raw = judge_row(row)
    except Exception as e:
        print(f"  [ERROR] id={row.get('id','?')}: {e}")
        score, raw = None, "ERROR"

    results.append({
        "id":                      row["id"],
        f"score_{MODEL_KEY}":      score,
        f"raw_{MODEL_KEY}":        raw,
        f"parse_fail_{MODEL_KEY}": score is None,
    })

    if (i + 1) % 25 == 0:
        elapsed = time.time() - t0
        valid   = [r[f"score_{MODEL_KEY}"] for r in results if r[f"score_{MODEL_KEY}"] is not None]
        avg     = sum(valid)/len(valid) if valid else 0
        print(f"  {i+1}/{len(df)} | {elapsed:.0f}s | avg={avg:.3f} | fails={sum(1 for r in results if r[f'parse_fail_{MODEL_KEY}'])}")
        pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)

# ── SAVE + SUMMARY ────────────────────────────────────────────────
pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
valid = [r[f"score_{MODEL_KEY}"] for r in results if r[f"score_{MODEL_KEY}"] is not None]
print(f"\n{'='*50}")
print(f"Model    : {MODEL_KEY} | Rows: {len(df)}")
print(f"Mean     : {sum(valid)/len(valid):.4f}")
print(f"Correct  : {sum(1 for s in valid if s==1.0)} ({sum(1 for s in valid if s==1.0)/len(valid)*100:.1f}%)")
print(f"Partial  : {sum(1 for s in valid if s==0.5)} ({sum(1 for s in valid if s==0.5)/len(valid)*100:.1f}%)")
print(f"Incorrect: {sum(1 for s in valid if s==0.0)} ({sum(1 for s in valid if s==0.0)/len(valid)*100:.1f}%)")

# ── UNLOAD ────────────────────────────────────────────────────────
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print("GPU cleared.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Free VRAM: 1.03 GB
Test Q : What is personal data under DPDP Act?
Test GT: Any data about an individual who is identifiable by or in relation to 
Score  : 0.0 | Raw: 0.0
  25/340 | 48s | avg=0.320 | fails=0
  50/340 | 97s | avg=0.360 | fails=0
  75/340 | 148s | avg=0.373 | fails=0
  100/340 | 202s | avg=0.420 | fails=0
  125/340 | 257s | avg=0.412 | fails=0
  150/340 | 308s | avg=0.377 | fails=0
  175/340 | 353s | avg=0.334 | fails=0
  200/340 | 401s | avg=0.323 | fails=0
  225/340 | 452s | avg=0.342 | fails=0
  250/340 | 504s | avg=0.340 | fails=0
  275/340 | 557s | avg=0.340 | fails=0
  300/340 | 612s | avg=0.333 | fails=0
  325/340 | 662s | avg=0.320 | fails=0

Model    : mistral | Rows: 340
Mean     : 0.3103
Correct  : 77 (22.6%)
Partial  : 57 (16.8%)
Incorrect: 206 (60.6%)
GPU cleared.


In [12]:
# run this to confirm
df = pd.read_csv(INPUT_CSV)
print(df.iloc[0]["system_answer"][:300])

[Dpdp Act | Section 1 — The Digital Personal Data Protection Act, 2023] THE DIGITAL PERSONAL DATA PROTECTION ACT, 2023 (NO. 22 OF 2023) [11th August, 2023.] An Act to provide for the processing of digital personal data in a manner that recognises both the right of individuals to protect their person


In [13]:
df = pd.read_csv(INPUT_CSV)
print(df["routed_engine"].value_counts())
print(f"\nTotal: {len(df)}")

routed_engine
Deep Thinker    255
Fast RAG         85
Name: count, dtype: int64

Total: 340


In [14]:
import os
os.remove("/kaggle/working/judge_qwen.csv")
print("Deleted. Now re-run the main cell.")

Deleted. Now re-run the main cell.
